# Custom Query Retrieval Probe

Use this notebook for one-off query analysis.

What it does:
- runs BGE-M3 dense retrieval
- runs BGE-M3 hybrid retrieval
- runs reranker sweeps for your custom query
- optionally compares with the OpenAI baseline
- shows whether an expected answer page/source is surfaced in top-k and promoted into top-3

To evaluate promotion into top-3, fill in at least one of these:
- `EXPECTED_PAGE_START` and `EXPECTED_PAGE_END`
- `EXPECTED_SOURCE_ID`


In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'packages' / 'rag_core').exists() and (candidate / 'apps' / 'api').exists():
            return candidate
    raise RuntimeError('Could not resolve repo root from notebook location.')

REPO_ROOT = resolve_repo_root(Path.cwd())
DOTENV_PATH = REPO_ROOT / 'docker' / '.env'
if DOTENV_PATH.exists():
    for raw_line in DOTENV_PATH.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

sys.path.insert(0, str(REPO_ROOT / 'packages' / 'rag_core' / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'apps' / 'api' / 'src'))

from rag_core.impl.embeddings_bge_m3 import BGEM3Embedder
from rag_core.impl.embeddings_openai import OpenAIEmbedder
from rag_core.impl.reranker_onnx import ONNXSequenceClassificationReranker
from rag_core.impl.vector_faiss import FaissVectorStore
from rag_core.rag.retrieval import DenseRetriever, LexicalRetriever, PageRetrievalEngine, RRFFusionPolicy


In [2]:
QUERY_TEXT =  "موضوع هجدهم کتاب دری صنف یازدهم چیست؟" 
QUERY_LANGUAGE = 'auto'

RUN_OPENAI_BASELINE = False
RUN_RERANKER = True

TOP_K = 10
RERANKER_TARGET_TOP_K = 3
RERANKER_CANDIDATE_POOL_SIZES = (3, 5, 8, 10, 15, 20)
DEFAULT_RERANKER_CANDIDATE_POOL_SIZE = 8

RRF_K = 20
DENSE_MIN_SCORE = 0.15
LEXICAL_MIN_SCORE = 0.01

EXPECTED_PAGE_START = None
EXPECTED_PAGE_END = None
EXPECTED_SOURCE_ID = ''
EXPECTED_ANSWER_TEXT = ''

BGE_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_bge_m3_full'
OPENAI_INDEX_DIR = REPO_ROOT / 'data' / 'index' / 'kankor_openai_full'

RERANKER_MODEL_ID = os.getenv('RAG_RERANKER_MODEL_ID', 'onnx-community/gte-multilingual-reranker-base')
RERANKER_MODEL_REVISION = os.getenv('RAG_RERANKER_MODEL_REVISION') or None
RERANKER_MAX_LENGTH = int(os.getenv('RAG_RERANKER_MAX_LENGTH', '256'))
RERANKER_BATCH_SIZE = int(os.getenv('RAG_RERANKER_BATCH_SIZE', '8'))

assert QUERY_TEXT.strip(), 'Set QUERY_TEXT before running the probe.'
assert (BGE_INDEX_DIR / 'index.faiss').exists(), f'Missing BGE index: {BGE_INDEX_DIR / "index.faiss"}'
assert (BGE_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing BGE metadata: {BGE_INDEX_DIR / "metadata.jsonl"}'
if RUN_OPENAI_BASELINE:
    assert (OPENAI_INDEX_DIR / 'index.faiss').exists(), f'Missing OpenAI index: {OPENAI_INDEX_DIR / "index.faiss"}'
    assert (OPENAI_INDEX_DIR / 'metadata.jsonl').exists(), f'Missing OpenAI metadata: {OPENAI_INDEX_DIR / "metadata.jsonl"}'

display({
    'query_text': QUERY_TEXT,
    'query_language': QUERY_LANGUAGE,
    'top_k': TOP_K,
    'run_openai_baseline': RUN_OPENAI_BASELINE,
    'run_reranker': RUN_RERANKER,
    'reranker_target_top_k': RERANKER_TARGET_TOP_K,
    'reranker_candidate_pool_sizes': RERANKER_CANDIDATE_POOL_SIZES,
    'default_reranker_candidate_pool_size': DEFAULT_RERANKER_CANDIDATE_POOL_SIZE,
    'expected_page_start': EXPECTED_PAGE_START,
    'expected_page_end': EXPECTED_PAGE_END,
    'expected_source_id': EXPECTED_SOURCE_ID,
    'bge_index_dir': str(BGE_INDEX_DIR),
})


{'query_text': 'موضوع هجدهم کتاب دری صنف یازدهم چیست؟',
 'query_language': 'auto',
 'top_k': 10,
 'run_openai_baseline': False,
 'run_reranker': True,
 'reranker_target_top_k': 3,
 'reranker_candidate_pool_sizes': (3, 5, 8, 10, 15, 20),
 'default_reranker_candidate_pool_size': 8,
 'expected_page_start': None,
 'expected_page_end': None,
 'expected_source_id': '',
 'bge_index_dir': '/home/nasher/Documents/projects/kankor-rag-space/data/index/kankor_bge_m3_full'}

In [3]:
def build_stack(index_dir: Path, *, embedder):
    vector_store = FaissVectorStore.load(
        index_path=index_dir / 'index.faiss',
        metadata_path=index_dir / 'metadata.jsonl',
    )
    dense = DenseRetriever(embedder=embedder, vector_store=vector_store, min_score=DENSE_MIN_SCORE)
    lexical = LexicalRetriever(vector_store=vector_store, min_score=LEXICAL_MIN_SCORE)
    engine = PageRetrievalEngine(
        dense_retriever=dense,
        lexical_retriever=lexical,
        fusion_policy=RRFFusionPolicy(rrf_k=RRF_K),
    )
    return vector_store, dense, engine

bge_embedder = BGEM3Embedder(
    model_name=os.getenv('RAG_BGE_M3_MODEL_ID', 'BAAI/bge-m3'),
    batch_size=int(os.getenv('RAG_BGE_M3_BATCH_SIZE', '32')),
    use_fp16=(os.getenv('RAG_BGE_M3_USE_FP16') or '').strip().lower() == 'true' if os.getenv('RAG_BGE_M3_USE_FP16') is not None else None,
    device=os.getenv('RAG_BGE_M3_DEVICE') or None,
    max_length=int(os.getenv('RAG_BGE_M3_MAX_LENGTH', '8192')),
)
_, bge_dense_retriever, bge_hybrid_engine = build_stack(BGE_INDEX_DIR, embedder=bge_embedder)

openai_dense_retriever = None
openai_hybrid_engine = None
if RUN_OPENAI_BASELINE:
    openai_embedder = OpenAIEmbedder(
        model_name=os.getenv('RAG_OPENAI_EMBEDDING_MODEL_ID', 'text-embedding-3-large'),
        api_key=os.getenv('RAG_OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY'),
        base_url=os.getenv('RAG_OPENAI_BASE_URL') or None,
        dimensions=int(os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS')) if os.getenv('RAG_OPENAI_EMBEDDING_DIMENSIONS') else None,
        timeout_seconds=float(os.getenv('RAG_OPENAI_TIMEOUT_SECONDS', '120.0')),
    )
    _, openai_dense_retriever, openai_hybrid_engine = build_stack(OPENAI_INDEX_DIR, embedder=openai_embedder)

reranker = None
if RUN_RERANKER:
    reranker = ONNXSequenceClassificationReranker(
        model_name=RERANKER_MODEL_ID,
        model_revision=RERANKER_MODEL_REVISION,
        max_length=RERANKER_MAX_LENGTH,
        batch_size=RERANKER_BATCH_SIZE,
    )
    reranker.warmup()

display({'bge_stack_ready': True, 'openai_baseline_ready': RUN_OPENAI_BASELINE, 'reranker_ready': RUN_RERANKER})


/home/nasher/miniconda3/envs/kankor-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'bge_stack_ready': True,
 'openai_baseline_ready': False,
 'reranker_ready': True}

In [4]:
def _coerce_positive_int(value):
    if value is None:
        return None
    try:
        parsed = int(str(value).strip())
    except (TypeError, ValueError):
        return None
    return parsed if parsed > 0 else None

def _hit_page_span(hit):
    metadata = hit.document.metadata
    start_page = _coerce_positive_int(metadata.get('start_page', metadata.get('page')))
    end_page = _coerce_positive_int(metadata.get('end_page', metadata.get('page')))
    return start_page, end_page

def _page_overlap(hit_start, hit_end, expected_start, expected_end):
    if hit_start is None or hit_end is None or expected_start is None or expected_end is None:
        return False
    return hit_end >= expected_start and hit_start <= expected_end

def hit_matches_expectation(hit):
    if not EXPECTATION_CONFIGURED:
        return None
    metadata = hit.document.metadata
    expected_start = _coerce_positive_int(EXPECTED_PAGE_START)
    expected_end = _coerce_positive_int(EXPECTED_PAGE_END)
    expected_source_id = str(EXPECTED_SOURCE_ID or '').strip()
    source_matches = True if not expected_source_id else str(metadata.get('source_id') or '') == expected_source_id
    if expected_start is None or expected_end is None:
        return source_matches if expected_source_id else False
    hit_start, hit_end = _hit_page_span(hit)
    return source_matches and _page_overlap(hit_start, hit_end, expected_start, expected_end)

def hits_to_frame(hits, *, system_name, query_text, latency_ms=None, reranker_candidate_pool_size=None):
    rows = []
    for rank, hit in enumerate(hits, start=1):
        metadata = hit.document.metadata
        start_page, end_page = _hit_page_span(hit)
        preview = ' '.join(str(hit.document.text or '').split())[:280]
        rows.append({
            'system': system_name,
            'query': query_text,
            'rank': rank,
            'score': float(hit.score),
            'relevant': hit_matches_expectation(hit),
            'source_id': str(metadata.get('source_id') or ''),
            'grade_band': str(metadata.get('grade_band') or ''),
            'subject': str(metadata.get('subject') or metadata.get('resolved_subject') or ''),
            'chapter_title': str(metadata.get('resolved_chapter_title') or metadata.get('chapter_title') or ''),
            'topic_title': str(metadata.get('resolved_topic_title') or metadata.get('topic_title') or ''),
            'start_page': start_page,
            'end_page': end_page,
            'latency_ms': latency_ms,
            'reranker_candidate_pool_size': reranker_candidate_pool_size,
            'text_preview': preview,
        })
    return pd.DataFrame(rows)

def first_relevant_rank(frame):
    if not EXPECTATION_CONFIGURED:
        return None
    if frame.empty or 'relevant' not in frame.columns:
        return None
    relevant_rows = frame.loc[frame['relevant']]
    if relevant_rows.empty:
        return None
    return int(relevant_rows['rank'].min())

def summarize_frame(frame, *, label, target_top_k=3):
    relevant_rank = first_relevant_rank(frame)
    return {
        'system': label,
        'expectation_configured': EXPECTATION_CONFIGURED,
        'manual_review_required': not EXPECTATION_CONFIGURED,
        'rows_returned': int(len(frame)),
        'first_relevant_rank': relevant_rank,
        f'relevant_in_top_{target_top_k}': (bool(relevant_rank is not None and relevant_rank <= target_top_k) if EXPECTATION_CONFIGURED else None),
        f'relevant_in_top_{TOP_K}': (bool(relevant_rank is not None and relevant_rank <= TOP_K) if EXPECTATION_CONFIGURED else None),
        'latency_ms': float(frame['latency_ms'].iloc[0]) if not frame.empty and pd.notna(frame['latency_ms'].iloc[0]) else None,
    }

EXPECTATION_CONFIGURED = any([
    _coerce_positive_int(EXPECTED_PAGE_START) is not None,
    _coerce_positive_int(EXPECTED_PAGE_END) is not None,
    str(EXPECTED_SOURCE_ID or '').strip(),
])

display({'expectation_configured': EXPECTATION_CONFIGURED, 'expected_answer_text': EXPECTED_ANSWER_TEXT})


{'expectation_configured': False, 'expected_answer_text': ''}

In [5]:
normalized_pool_sizes = tuple(sorted({max(RERANKER_TARGET_TOP_K, int(value)) for value in RERANKER_CANDIDATE_POOL_SIZES}))

dense_started = time.perf_counter()
bge_dense_hits, bge_dense_search_calls = bge_dense_retriever.search_many(
    questions=[QUERY_TEXT],
    top_k=TOP_K,
)
bge_dense_ms = int((time.perf_counter() - dense_started) * 1000)
bge_dense_frame = hits_to_frame(bge_dense_hits[:TOP_K], system_name='bge_dense', query_text=QUERY_TEXT, latency_ms=bge_dense_ms)

bge_hybrid_hits, bge_hybrid_diag = bge_hybrid_engine.search(
    questions=[QUERY_TEXT],
    top_k=TOP_K,
)
bge_hybrid_frame = hits_to_frame(bge_hybrid_hits[:TOP_K], system_name='bge_hybrid', query_text=QUERY_TEXT, latency_ms=bge_hybrid_diag.total_ms)

openai_dense_frame = None
openai_hybrid_frame = None
if RUN_OPENAI_BASELINE:
    openai_dense_started = time.perf_counter()
    openai_dense_hits, _ = openai_dense_retriever.search_many(questions=[QUERY_TEXT], top_k=TOP_K)
    openai_dense_ms = int((time.perf_counter() - openai_dense_started) * 1000)
    openai_dense_frame = hits_to_frame(openai_dense_hits[:TOP_K], system_name='openai_dense', query_text=QUERY_TEXT, latency_ms=openai_dense_ms)

    openai_hybrid_hits, openai_hybrid_diag = openai_hybrid_engine.search(questions=[QUERY_TEXT], top_k=TOP_K)
    openai_hybrid_frame = hits_to_frame(openai_hybrid_hits[:TOP_K], system_name='openai_hybrid', query_text=QUERY_TEXT, latency_ms=openai_hybrid_diag.total_ms)

bge_reranked_frames_by_pool = {}
reranker_probe_rows = []
if RUN_RERANKER:
    for candidate_pool_size in normalized_pool_sizes:
        fused_hits, fused_diag = bge_hybrid_engine.search(
            questions=[QUERY_TEXT],
            top_k=max(TOP_K, candidate_pool_size),
        )
        rerank_started = time.perf_counter()
        reranked_hits = reranker.rerank(
            query=QUERY_TEXT,
            hits=fused_hits,
            top_k=max(TOP_K, RERANKER_TARGET_TOP_K),
        )
        rerank_ms = int((time.perf_counter() - rerank_started) * 1000)
        total_ms = fused_diag.total_ms + rerank_ms
        frame = hits_to_frame(
            reranked_hits[:TOP_K],
            system_name=f'bge_reranked_pool_{candidate_pool_size}',
            query_text=QUERY_TEXT,
            latency_ms=total_ms,
            reranker_candidate_pool_size=candidate_pool_size,
        )
        bge_reranked_frames_by_pool[candidate_pool_size] = frame

        hybrid_pool_frame = hits_to_frame(
            fused_hits[:candidate_pool_size],
            system_name=f'bge_hybrid_pool_{candidate_pool_size}',
            query_text=QUERY_TEXT,
            latency_ms=fused_diag.total_ms,
            reranker_candidate_pool_size=candidate_pool_size,
        )
        hybrid_pool_rank = first_relevant_rank(hybrid_pool_frame)
        reranked_rank = first_relevant_rank(frame)
        reranker_probe_rows.append({
            'candidate_pool_size': candidate_pool_size,
            'hybrid_latency_ms': fused_diag.total_ms,
            'rerank_only_ms': rerank_ms,
            'end_to_end_ms': total_ms,
            'hybrid_first_relevant_rank_in_pool': hybrid_pool_rank,
            'reranked_first_relevant_rank': reranked_rank,
            'relevant_in_hybrid_pool': bool(hybrid_pool_rank is not None and hybrid_pool_rank <= candidate_pool_size),
            'relevant_in_reranked_top3': bool(reranked_rank is not None and reranked_rank <= RERANKER_TARGET_TOP_K),
            'promoted_into_top3': bool(
                hybrid_pool_rank is not None
                and hybrid_pool_rank > RERANKER_TARGET_TOP_K
                and reranked_rank is not None
                and reranked_rank <= RERANKER_TARGET_TOP_K
            ),
        })

summary_rows = [
    summarize_frame(bge_dense_frame, label='bge_dense', target_top_k=RERANKER_TARGET_TOP_K),
    summarize_frame(bge_hybrid_frame, label='bge_hybrid', target_top_k=RERANKER_TARGET_TOP_K),
]
if RUN_RERANKER:
    default_frame = bge_reranked_frames_by_pool[DEFAULT_RERANKER_CANDIDATE_POOL_SIZE]
    summary_rows.append(
        summarize_frame(
            default_frame,
            label=f'bge_reranked_pool_{DEFAULT_RERANKER_CANDIDATE_POOL_SIZE}',
            target_top_k=RERANKER_TARGET_TOP_K,
        )
    )
if openai_dense_frame is not None:
    summary_rows.append(summarize_frame(openai_dense_frame, label='openai_dense', target_top_k=RERANKER_TARGET_TOP_K))
if openai_hybrid_frame is not None:
    summary_rows.append(summarize_frame(openai_hybrid_frame, label='openai_hybrid', target_top_k=RERANKER_TARGET_TOP_K))

summary_frame = pd.DataFrame(summary_rows)
reranker_probe_frame = pd.DataFrame(reranker_probe_rows)

display(summary_frame)
if RUN_RERANKER:
    display(reranker_probe_frame)


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 5713.27it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,system,expectation_configured,manual_review_required,rows_returned,first_relevant_rank,relevant_in_top_3,relevant_in_top_10,latency_ms
0,bge_dense,False,True,10,None,None,None,7984.0
1,bge_hybrid,False,True,10,None,None,None,36.0
2,bge_reranked_pool_8,False,True,10,None,None,None,2240.0


,candidate_pool_size,hybrid_latency_ms,rerank_only_ms,end_to_end_ms,hybrid_first_relevant_rank_in_pool,reranked_first_relevant_rank,relevant_in_hybrid_pool,relevant_in_reranked_top3,promoted_into_top3
0,3,35,2387,2422,None,None,False,False,False
1,5,44,2579,2623,None,None,False,False,False
2,8,47,2193,2240,None,None,False,False,False
3,10,44,1945,1989,None,None,False,False,False
4,15,45,3068,3113,None,None,False,False,False
5,20,47,4502,4549,None,None,False,False,False


In [6]:
display(bge_dense_frame)
display(bge_hybrid_frame)

if RUN_RERANKER:
    for candidate_pool_size, frame in sorted(bge_reranked_frames_by_pool.items()):
        print(f'Reranked results for candidate_pool_size={candidate_pool_size}')
        display(frame)

if openai_dense_frame is not None:
    display(openai_dense_frame)
if openai_hybrid_frame is not None:
    display(openai_hybrid_frame)


,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,7984,None,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,7984,None,book G10-Dr-Dari | subject dari | grade 10 cha...
2,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,7984,None,book G11-Dr-Dari | subject dari | grade 11 cha...
3,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.549779,None,G12-Dr-Islamic_Study_Jafari,12,islamic_studies,بهشت و جهنم,بهشت و جهنم,37,37,7984,None,book G12-Dr-Islamic_Study_Jafari | subject isl...
4,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.546833,None,G10-Dr-Dari,10,dari,تفاهم و همدیگر پذیری,تفاهم و همدیگر پذیری,84,84,7984,None,book G10-Dr-Dari | subject dari | grade 10 cha...
5,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.546426,None,G11-Dr-Dari,11,dari,سالار پیامبران,سالار پیامبران,7,7,7984,None,book G11-Dr-Dari | subject dari | grade 11 cha...
6,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.544828,None,G12-Dr-Tafseer,12,tafseer,ادامه موضوع گذشته,ادامه موضوع گذشته,76,76,7984,None,book G12-Dr-Tafseer | subject tafseer | grade ...
7,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.544570,None,G11-Dr-Islamic_Study_Hanafi,11,islamic_studies,بخش عقاید,وحی,6,6,7984,None,book G11-Dr-Islamic_Study_Hanafi | subject isl...
8,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.541369,None,G11-Dr-Islamic_Study_Hanafi,11,islamic_studies,بخش عقاید,وحی,7,7,7984,None,book G11-Dr-Islamic_Study_Hanafi | subject isl...
9,bge_dense,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.537150,None,G11-Dr-Dari,11,dari,فارسی یا دری,فارسی یا دری,60,60,7984,None,book G11-Dr-Dari | subject dari | grade 11 cha...


,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,36,None,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.535714,None,G12-Dr-Computer,12,computer_science,فصل هشتم: آشنایی با مرور کننده ها (Browsers),درس بیست و پنجم مرور کننده (Browser) چیست؟,100,100,36,None,book G12-Dr-Computer | subject computer_scienc...
2,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,36,None,book G10-Dr-Dari | subject dari | grade 10 cha...
3,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.428571,None,G10-Dr-Computer,10,computer_science,کمپیوتر و کاربرد آن,کمپیوتر چیست؟,4,4,36,None,book G10-Dr-Computer | subject computer_scienc...
4,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,36,None,book G11-Dr-Dari | subject dari | grade 11 cha...
5,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.428571,None,G10-Dr-physic,10,physics,فزیک چیست,تاریخچه مختصر فزیک,4,4,36,None,book G10-Dr-physic | subject physics | grade 1...
6,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.549779,None,G12-Dr-Islamic_Study_Jafari,12,islamic_studies,بهشت و جهنم,بهشت و جهنم,37,37,36,None,book G12-Dr-Islamic_Study_Jafari | subject isl...
7,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.428571,None,G10-Dr-physic,10,physics,اندازه گیری اندازه گیری چیست؟,اندازه گیری اندازه گیری چیست؟,13,13,36,None,book G10-Dr-physic | subject physics | grade 1...
8,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.546833,None,G10-Dr-Dari,10,dari,تفاهم و همدیگر پذیری,تفاهم و همدیگر پذیری,84,84,36,None,book G10-Dr-Dari | subject dari | grade 10 cha...
9,bge_hybrid,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.428571,None,G11-Dr-Biology,11,biology,مطالعه حجره و انواع میکروسکوپها,مطالعه حجره و انواع میکروسکوپها,4,4,36,None,book G11-Dr-Biology | subject biology | grade ...


Reranked results for candidate_pool_size=3


,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,2422,3,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,2422,3,book G11-Dr-Dari | subject dari | grade 11 cha...
2,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,2422,3,book G10-Dr-Dari | subject dari | grade 10 cha...
3,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.535714,None,G12-Dr-Computer,12,computer_science,فصل هشتم: آشنایی با مرور کننده ها (Browsers),درس بیست و پنجم مرور کننده (Browser) چیست؟,100,100,2422,3,book G12-Dr-Computer | subject computer_scienc...
4,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.428571,None,G11-Dr-Biology,11,biology,مطالعه حجره و انواع میکروسکوپها,مطالعه حجره و انواع میکروسکوپها,4,4,2422,3,book G11-Dr-Biology | subject biology | grade ...
5,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.546833,None,G10-Dr-Dari,10,dari,تفاهم و همدیگر پذیری,تفاهم و همدیگر پذیری,84,84,2422,3,book G10-Dr-Dari | subject dari | grade 10 cha...
6,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.549779,None,G12-Dr-Islamic_Study_Jafari,12,islamic_studies,بهشت و جهنم,بهشت و جهنم,37,37,2422,3,book G12-Dr-Islamic_Study_Jafari | subject isl...
7,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.428571,None,G10-Dr-physic,10,physics,فزیک چیست,تاریخچه مختصر فزیک,4,4,2422,3,book G10-Dr-physic | subject physics | grade 1...
8,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.428571,None,G10-Dr-physic,10,physics,اندازه گیری اندازه گیری چیست؟,اندازه گیری اندازه گیری چیست؟,13,13,2422,3,book G10-Dr-physic | subject physics | grade 1...
9,bge_reranked_pool_3,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.428571,None,G10-Dr-Computer,10,computer_science,کمپیوتر و کاربرد آن,کمپیوتر چیست؟,4,4,2422,3,book G10-Dr-Computer | subject computer_scienc...


Reranked results for candidate_pool_size=5


...

,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,1989,10,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,1989,10,book G11-Dr-Dari | subject dari | grade 11 cha...
2,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,1989,10,book G10-Dr-Dari | subject dari | grade 10 cha...
3,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.535714,None,G12-Dr-Computer,12,computer_science,فصل هشتم: آشنایی با مرور کننده ها (Browsers),درس بیست و پنجم مرور کننده (Browser) چیست؟,100,100,1989,10,book G12-Dr-Computer | subject computer_scienc...
4,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.428571,None,G11-Dr-Biology,11,biology,مطالعه حجره و انواع میکروسکوپها,مطالعه حجره و انواع میکروسکوپها,4,4,1989,10,book G11-Dr-Biology | subject biology | grade ...
5,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.546833,None,G10-Dr-Dari,10,dari,تفاهم و همدیگر پذیری,تفاهم و همدیگر پذیری,84,84,1989,10,book G10-Dr-Dari | subject dari | grade 10 cha...
6,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.549779,None,G12-Dr-Islamic_Study_Jafari,12,islamic_studies,بهشت و جهنم,بهشت و جهنم,37,37,1989,10,book G12-Dr-Islamic_Study_Jafari | subject isl...
7,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.428571,None,G10-Dr-physic,10,physics,فزیک چیست,تاریخچه مختصر فزیک,4,4,1989,10,book G10-Dr-physic | subject physics | grade 1...
8,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.428571,None,G10-Dr-physic,10,physics,اندازه گیری اندازه گیری چیست؟,اندازه گیری اندازه گیری چیست؟,13,13,1989,10,book G10-Dr-physic | subject physics | grade 1...
9,bge_reranked_pool_10,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.428571,None,G10-Dr-Computer,10,computer_science,کمپیوتر و کاربرد آن,کمپیوتر چیست؟,4,4,1989,10,book G10-Dr-Computer | subject computer_scienc...


Reranked results for candidate_pool_size=15


,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,3113,15,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,3113,15,book G11-Dr-Dari | subject dari | grade 11 cha...
2,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,3113,15,book G10-Dr-Dari | subject dari | grade 10 cha...
3,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.428571,None,G11-Dr-Computer,11,computer_science,فصل اول,درس دوم: پروگرامهای محاورهیی کمپیوتر,4,4,3113,15,book G11-Dr-Computer | subject computer_scienc...
4,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.428571,None,G11-Dr-Biology,11,biology,مطالعه حجره و انواع میکروسکوپها,مطالعه حجره و انواع میکروسکوپها,4,4,3113,15,book G11-Dr-Biology | subject biology | grade ...
5,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.535714,None,G12-Dr-Computer,12,computer_science,فصل هشتم: آشنایی با مرور کننده ها (Browsers),درس بیست و پنجم مرور کننده (Browser) چیست؟,100,100,3113,15,book G12-Dr-Computer | subject computer_scienc...
6,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.428571,None,G11-Dr-CIvic,11,civic_education,,,4,4,3113,15,book G11-Dr-CIvic | subject civic_education | ...
7,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.546833,None,G10-Dr-Dari,10,dari,تفاهم و همدیگر پذیری,تفاهم و همدیگر پذیری,84,84,3113,15,book G10-Dr-Dari | subject dari | grade 10 cha...
8,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.544828,None,G12-Dr-Tafseer,12,tafseer,ادامه موضوع گذشته,ادامه موضوع گذشته,76,76,3113,15,book G12-Dr-Tafseer | subject tafseer | grade ...
9,bge_reranked_pool_15,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.544570,None,G11-Dr-Islamic_Study_Hanafi,11,islamic_studies,بخش عقاید,وحی,6,6,3113,15,book G11-Dr-Islamic_Study_Hanafi | subject isl...


Reranked results for candidate_pool_size=20


,system,query,rank,score,relevant,source_id,grade_band,subject,chapter_title,topic_title,start_page,end_page,latency_ms,reranker_candidate_pool_size,text_preview
0,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,1,0.558052,None,G11-Dr-Tafseer,11,tafseer,فهرست,درس نوزدهم: منزلت مؤمنان معیوب و فقیر در اسلام,89,89,4549,20,book G11-Dr-Tafseer | subject tafseer | grade ...
1,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,2,0.554292,None,G11-Dr-Dari,11,dari,اندرزهای اخلاقی,اندرزهای اخلاقی,96,96,4549,20,book G11-Dr-Dari | subject dari | grade 11 cha...
2,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,3,0.555282,None,G10-Dr-Dari,10,dari,بلخ باستان,بلخ باستان,88,88,4549,20,book G10-Dr-Dari | subject dari | grade 10 cha...
3,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,4,0.428571,None,G11-Dr-Computer,11,computer_science,فصل اول,درس دوم: پروگرامهای محاورهیی کمپیوتر,4,4,4549,20,book G11-Dr-Computer | subject computer_scienc...
4,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,5,0.428571,None,G11-Dr-Biology,11,biology,مطالعه حجره و انواع میکروسکوپها,مطالعه حجره و انواع میکروسکوپها,4,4,4549,20,book G11-Dr-Biology | subject biology | grade ...
5,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,6,0.537150,None,G11-Dr-Dari,11,dari,فارسی یا دری,فارسی یا دری,60,60,4549,20,book G11-Dr-Dari | subject dari | grade 11 cha...
6,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,7,0.428571,None,G11-Dr-Dari,11,dari,حمد,حمد,4,4,4549,20,book G11-Dr-Dari | subject dari | grade 11 cha...
7,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,8,0.428571,None,G11-Dr-History,11,history,فهرست مطالب,اوضاع عمومی افغانستان مقارن ظهور اسلام,4,4,4549,20,book G11-Dr-History | subject history | grade ...
8,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,9,0.428571,None,G11-Dr-CIvic,11,civic_education,,,4,4,4549,20,book G11-Dr-CIvic | subject civic_education | ...
9,bge_reranked_pool_20,موضوع هجدهم کتاب دری صنف یازدهم چیست؟,10,0.535714,None,G12-Dr-Computer,12,computer_science,فصل هشتم: آشنایی با مرور کننده ها (Browsers),درس بیست و پنجم مرور کننده (Browser) چیست؟,100,100,4549,20,book G12-Dr-Computer | subject computer_scienc...


## Notes

If `EXPECTATION_CONFIGURED` is `False`, the notebook will still show retrieved pages and rankings, but it cannot automatically tell you whether the correct answer was promoted into top-3.

For the strongest check, set:
- `EXPECTED_PAGE_START`
- `EXPECTED_PAGE_END`
- optionally `EXPECTED_SOURCE_ID`
